# 04e Taiwan Policy Features (FIT)

將台灣各縣市歷年太陽光電躉購費率（FIT, Feed-in Tariff）整理為可供 Stage 2 transfer learning 使用的政策誘因特徵。

**定位**：縣市層級政策誘因 proxy（非個別案場 FIT）。

**DeepSolar 概念對應（重要差異）**：

| | 美國 DeepSolar `feedin_tariff` | 本專案台灣欄位 |
|---|---|---|
| 定義 | **Number of years since the start of feed-in tariff**（政策實施年數） | **實際躉購費率**（元/度，加成後 FIT） |
| 性質 | 政策存續時間 proxy | 政策誘因強度 proxy（費率越高，誘因越大） |
| 欄位名稱 | 單一 `feedin_tariff` | `{類型}_{容量級距}_加成後FIT（元/度）`（無法 1:1 對應，故保留結構化命名） |

**Input**：`data/taiwan/policy/taiwan_solar_fit_by_county_107_115_revised.xlsx`（主表 `FIT_by_County_Long`）

**Output**：`data/taiwan/policy/taiwan_policy_feature.csv`（368 鄉鎮 × 識別欄 + 各類型/級距 FIT 欄）

詳細交接說明見：`data/taiwan/policy/04e_taiwan_policy_summary.md`

## 資料說明與聚合邏輯

原始資料為**縣市層級**（22 縣市 × 民國 107–115 年 × 兩期 × 類型 × 容量級距）。輸出對齊其他台灣特徵表，以 `TOWNCODE` 為列，同一縣市內各鄉鎮複製相同 FIT 值。

**聚合步驟**（針對每個縣市 × 類型 × 容量級距）：
1. **期別平均**：同一年度內，將第一期與第二期 `加成後FIT(元/度)` 取平均
2. **107–112 級距對照**：舊制粗級距展開為 113 年制 8 種名稱（見下表）
3. **年度平均**：對民國 107–115 共 9 年取平均
4. **下放到鄉鎮**：以 `COUNTYNAME` 對齊人口表，同縣市各鄉鎮複製相同值

輸出欄位命名：`{類型}_{容量級距}_加成後FIT（元/度）`。

### 容量級距制度變更（107–112 vs 113–115）— **統一為 113 年制 8 種命名**

能源局自 **民國 113 年**起將屋頂型容量細分為 8 種「類型×級距」；112 年（含）以前每年僅 **6** 種。本 notebook **統一採 113 年制 8 欄**輸出：113–115 年保留原始細分級距；107–112 年將舊制粗級距**展開**對應至兩個新制名稱（同一 FIT 值計入兩欄的年度平均）。

| 107–112 舊制級距 | 展開為 113 年制（2 欄） |
|---|---|
| `1瓩以上不及20瓩` | `1瓩以上不及10瓩`、`10瓩以上不及20瓩` |
| `20瓩以上不及100瓩` | `20瓩以上不及50瓩`、`50瓩以上不及100瓩` |

其餘 4 種（地面型/水面型/屋頂型大容量）兩個時期名稱相同，無需展開。對照後同一年若有多筆（細分級距或兩期），先取算術平均，再對 **107–115 共 9 年**取平均。

In [1]:
# %%
# 1. Imports

import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

PROJ = Path('..').resolve()
POLICY_DIR = PROJ / 'data' / 'taiwan' / 'policy'
POP_CSV = PROJ / 'data' / 'taiwan' / 'population' / 'taiwan_population_features.csv'

OUT_CSV = POLICY_DIR / 'taiwan_policy_feature.csv'
FIT_SHEET = 'FIT_by_County_Long'

POLICY_DIR.mkdir(parents=True, exist_ok=True)

COUNTIES_22 = {
    '基隆市', '臺北市', '新北市', '桃園市', '新竹市', '新竹縣', '苗栗縣',
    '臺中市', '彰化縣', '南投縣', '雲林縣', '嘉義市', '嘉義縣', '臺南市',
    '高雄市', '屏東縣', '宜蘭縣', '花蓮縣', '臺東縣', '澎湖縣', '金門縣', '連江縣',
}

COLUMN_MAP = {
    '民國年': 'year_roc',
    '西元年': 'year',
    '縣市': 'county',
    '分類': 'solar_type',
    '裝置容量級距': 'capacity_category',
    '期別': 'phase',
    '加成後FIT(元/度)': 'feedin_tariff',
}

ID_COLS = ['TOWNCODE', 'COUNTYNAME', 'TOWNNAME']

print('Project root:', PROJ)
print('Policy dir:', POLICY_DIR)
print('Population ref:', POP_CSV.exists())

Project root: /Users/tingting/SolarPotentialMapping
Policy dir: /Users/tingting/SolarPotentialMapping/data/taiwan/policy
Population ref: True


In [2]:
def find_fit_source(policy_dir: Path) -> Path:
    """Locate FIT workbook under data/taiwan/policy."""
    patterns = ('*fit*county*.xlsx', '*FIT*county*.xlsx')
    candidates = []
    for pat in patterns:
        candidates.extend(sorted(policy_dir.glob(pat)))
    candidates = sorted({p.resolve() for p in candidates})
    if not candidates:
        raise FileNotFoundError(f'No FIT xlsx found under: {policy_dir}')
    preferred = [p for p in candidates if 'revised' in p.name.lower()]
    return preferred[0] if preferred else candidates[0]


def normalize_county_name(name) -> Optional[str]:
    """Standardize county names: 台→臺, strip whitespace, fix duplicated suffix."""
    if pd.isna(name):
        return None
    s = str(name).strip().replace('台', '臺')
    s = re.sub(r'\s+', '', s)
    for suffix in ('市', '縣'):
        if s.endswith(suffix * 2):
            s = s[:-len(suffix)]
    return s


def standardize_solar_type(solar_type: str) -> Optional[str]:
    if pd.isna(solar_type):
        return None
    s = str(solar_type).strip()
    if s.startswith('水面型'):
        return '水面型'
    return s


def make_fit_feature_col(solar_type: str, capacity_category: str) -> str:
    """Column name: {類型}_{級距}_加成後FIT（元/度）."""
    return f'{solar_type}_{capacity_category}_加成後FIT（元/度）'


# 107–112 舊制屋頂級距 → 展開為 113 年制細分名稱（一對二）
CAPACITY_EXPAND_FROM_OLD = {
    '1瓩以上不及20瓩': ['1瓩以上不及10瓩', '10瓩以上不及20瓩'],
    '20瓩以上不及100瓩': ['20瓩以上不及50瓩', '50瓩以上不及100瓩'],
}

UNIFIED_TIER_COUNT = 8  # 113 年制「類型×級距」組合數


def parse_numeric_series(s: pd.Series) -> pd.Series:
    if s.dtype.kind in 'biufc':
        return pd.to_numeric(s, errors='coerce')
    cleaned = (
        s.astype(str).str.strip()
        .replace({'': np.nan, 'nan': np.nan, 'None': np.nan, '-': np.nan})
    )
    has_pct = cleaned.str.contains('%', regex=False, na=False)
    cleaned = cleaned.str.replace(',', '', regex=False).str.replace('%', '', regex=False)
    out = pd.to_numeric(cleaned, errors='coerce')
    if has_pct.any():
        out = np.where(has_pct, out / 100.0, out)
    return pd.Series(out, index=s.index, dtype='float64')


def capacity_tier_coverage(fit_df: pd.DataFrame) -> pd.DataFrame:
    """Unified tier labels used in output (expect 8 combos, 9 years each)."""
    cov = (
        fit_df.groupby(['solar_type', 'capacity_category'])['year_roc']
        .apply(lambda s: sorted(s.unique()))
        .reset_index(name='years')
    )
    cov['feature_col'] = cov.apply(
        lambda r: make_fit_feature_col(r['solar_type'], r['capacity_category']), axis=1
    )
    cov['n_years'] = cov['years'].apply(len)
    return cov.sort_values('feature_col').reset_index(drop=True)


def apply_capacity_unification(fit_df: pd.DataFrame) -> pd.DataFrame:
    """Expand 107-112 coarse tiers to 113-era labels; 113+ rows unchanged."""
    out = fit_df.copy()
    out['capacity_category_raw'] = out['capacity_category']
    rows = []
    for _, row in out.iterrows():
        cap_raw = str(row['capacity_category_raw']).strip()
        if int(row['year_roc']) <= 112 and cap_raw in CAPACITY_EXPAND_FROM_OLD:
            for new_cap in CAPACITY_EXPAND_FROM_OLD[cap_raw]:
                r = row.copy()
                r['capacity_category'] = new_cap
                rows.append(r)
        else:
            r = row.copy()
            r['capacity_category'] = cap_raw
            rows.append(r)
    return pd.DataFrame(rows).reset_index(drop=True)


def aggregate_fit_to_county_features(fit_df: pd.DataFrame) -> pd.DataFrame:
    """Within year: mean across phases and mapped sub-tiers; then mean across 107-115."""
    year_avg = (
        fit_df
        .groupby(['county', 'year', 'solar_type', 'capacity_category'], as_index=False)['feedin_tariff']
        .mean()
    )
    county_features = (
        year_avg
        .groupby(['county', 'solar_type', 'capacity_category'], as_index=False)['feedin_tariff']
        .mean()
    )
    county_features['feature_col'] = county_features.apply(
        lambda r: make_fit_feature_col(r['solar_type'], r['capacity_category']), axis=1
    )
    return county_features


def pivot_county_fit_features(county_features: pd.DataFrame) -> pd.DataFrame:
    wide = (
        county_features
        .pivot(index='county', columns='feature_col', values='feedin_tariff')
        .reset_index()
        .rename(columns={'county': 'COUNTYNAME'})
    )
    fit_cols = sorted([c for c in wide.columns if c.endswith('加成後FIT（元/度）')])
    return wide[['COUNTYNAME'] + fit_cols]


def expand_county_to_township(county_wide: pd.DataFrame, pop_ref: pd.DataFrame) -> pd.DataFrame:
    """Replicate county-level FIT to all townships (same pattern as electricity features)."""
    out = (
        pop_ref[ID_COLS]
        .merge(county_wide, on='COUNTYNAME', how='left')
        .sort_values('TOWNCODE')
        .reset_index(drop=True)
    )
    return out

## 2. Load raw FIT data

In [3]:
# %%
# 2. Load raw FIT data

FIT_PATH = find_fit_source(POLICY_DIR)
print('FIT source:', FIT_PATH)

xl = pd.ExcelFile(FIT_PATH)
print('Sheets:', xl.sheet_names)
assert FIT_SHEET in xl.sheet_names

fit_raw = pd.read_excel(FIT_PATH, sheet_name=FIT_SHEET)
print('Raw shape:', fit_raw.shape)
fit_raw.head(3)

FIT source: /Users/tingting/SolarPotentialMapping/data/taiwan/policy/taiwan_solar_fit_by_county_107_115_revised.xlsx
Sheets: ['FIT_by_County_Long', 'Base_FIT_Official', 'County_Adders', 'Sources', 'Method_Notes']


Raw shape: (2860, 16)


,西元年,民國年,期別,縣市,縣市分類,分類,裝置容量級距,條件/備註,基礎費率(元/度),縣市/離島加成比例,加成後FIT(元/度),VPC高效模組加成比例,含VPC高效模組FIT(元/度),離島替代加成比例,來源,計算備註
0,2018,107,第一期,基隆市,北部/東北部加成,屋頂型,1瓩以上不及20瓩,NaN,5.8744,0.15,6.7556,0.06,7.108,NaN,附件PDF：附表三 107 年度太陽光電發電設備電能躉購費率.pdf,依使用者提供CSV：107~115年15%
1,2018,107,第一期,臺北市,北部/東北部加成,屋頂型,1瓩以上不及20瓩,NaN,5.8744,0.15,6.7556,0.06,7.108,NaN,附件PDF：附表三 107 年度太陽光電發電設備電能躉購費率.pdf,依使用者提供CSV：107~115年15%
2,2018,107,第一期,新北市,北部/東北部加成,屋頂型,1瓩以上不及20瓩,NaN,5.8744,0.15,6.7556,0.06,7.108,NaN,附件PDF：附表三 107 年度太陽光電發電設備電能躉購費率.pdf,依使用者提供CSV：107~115年15%


## 3. Basic cleaning

In [4]:
# %%
# 3. Basic cleaning

fit_clean = (
    fit_raw
    .rename(columns=COLUMN_MAP)
    .loc[:, COLUMN_MAP.values()]
    .assign(
        year_roc=lambda d: parse_numeric_series(d['year_roc']).astype('Int64'),
        year=lambda d: parse_numeric_series(d['year_roc']) + 1911,
        feedin_tariff=lambda d: parse_numeric_series(d['feedin_tariff']),
        solar_type=lambda d: d['solar_type'].map(standardize_solar_type),
        capacity_category=lambda d: d['capacity_category'].astype(str).str.strip(),
        county=lambda d: d['county'].map(normalize_county_name),
    )
)
fit_clean = apply_capacity_unification(fit_clean)

print('Cleaned shape:', fit_clean.shape)
print('Type × capacity combos:', fit_clean.groupby(['solar_type', 'capacity_category']).ngroups)
fit_clean.head(3)

Cleaned shape: (3476, 8)


Type × capacity combos: 8


,year_roc,year,county,solar_type,capacity_category,phase,feedin_tariff,capacity_category_raw
0,107,2018,基隆市,屋頂型,1瓩以上不及10瓩,第一期,6.7556,1瓩以上不及20瓩
1,107,2018,基隆市,屋頂型,10瓩以上不及20瓩,第一期,6.7556,1瓩以上不及20瓩
2,107,2018,臺北市,屋頂型,1瓩以上不及10瓩,第一期,6.7556,1瓩以上不及20瓩


## 4. County normalization check

In [5]:
# %%
# 4. County normalization

unknown = sorted(set(fit_clean['county'].dropna()) - COUNTIES_22)
missing = sorted(COUNTIES_22 - set(fit_clean['county'].dropna()))
print('Counties in data:', fit_clean['county'].nunique())
print('Unknown labels:', unknown or '(none)')
print('Missing from 22:', missing or '(none)')

Counties in data: 22
Unknown labels: (none)
Missing from 22: (none)


## 5. Feature engineering

In [6]:
# %%
# 5. Feature engineering — phase avg → year avg → county wide → township expand

print('=== 107-112 舊制 → 113 年制 8 欄展開對照 ===')
for old_cap, new_caps in CAPACITY_EXPAND_FROM_OLD.items():
    print(f'  {old_cap}  →  {new_caps}')

tier_coverage = capacity_tier_coverage(fit_clean)
print('\n=== Unified output tiers (expect 8 × 9 years) ===')
print(tier_coverage[['feature_col', 'n_years']].to_string(index=False))
assert len(tier_coverage) == UNIFIED_TIER_COUNT, (
    f'Expected {UNIFIED_TIER_COUNT} unified tiers, got {len(tier_coverage)}'
)
assert tier_coverage['n_years'].eq(9).all(), 'Each tier should span ROC 107-115'

county_long = aggregate_fit_to_county_features(fit_clean)
county_wide = pivot_county_fit_features(county_long)

pop_ref = pd.read_csv(POP_CSV, encoding='utf-8-sig')[ID_COLS]
policy_features = expand_county_to_township(county_wide, pop_ref)

fit_feature_cols = [c for c in policy_features.columns if c.endswith('加成後FIT（元/度）')]

print('County-level feature rows:', len(county_wide))
print('Township output shape:', policy_features.shape)
print(f'FIT feature columns ({len(fit_feature_cols)}):')
for c in fit_feature_cols:
    print(' ', c)
policy_features.head(3)

=== 107-112 舊制 → 113 年制 8 欄展開對照 ===
  1瓩以上不及20瓩  →  ['1瓩以上不及10瓩', '10瓩以上不及20瓩']
  20瓩以上不及100瓩  →  ['20瓩以上不及50瓩', '50瓩以上不及100瓩']

=== Unified output tiers (expect 8 × 9 years) ===
                 feature_col  n_years
        地面型_1瓩以上_加成後FIT（元/度）        9
屋頂型_100瓩以上不及500瓩_加成後FIT（元/度）        9
  屋頂型_10瓩以上不及20瓩_加成後FIT（元/度）        9
   屋頂型_1瓩以上不及10瓩_加成後FIT（元/度）        9
  屋頂型_20瓩以上不及50瓩_加成後FIT（元/度）        9
      屋頂型_500瓩以上_加成後FIT（元/度）        9
 屋頂型_50瓩以上不及100瓩_加成後FIT（元/度）        9
        水面型_1瓩以上_加成後FIT（元/度）        9
County-level feature rows: 22
Township output shape: (368, 11)
FIT feature columns (8):
  地面型_1瓩以上_加成後FIT（元/度）
  屋頂型_100瓩以上不及500瓩_加成後FIT（元/度）
  屋頂型_10瓩以上不及20瓩_加成後FIT（元/度）
  屋頂型_1瓩以上不及10瓩_加成後FIT（元/度）
  屋頂型_20瓩以上不及50瓩_加成後FIT（元/度）
  屋頂型_500瓩以上_加成後FIT（元/度）
  屋頂型_50瓩以上不及100瓩_加成後FIT（元/度）
  水面型_1瓩以上_加成後FIT（元/度）


,TOWNCODE,COUNTYNAME,TOWNNAME,地面型_1瓩以上_加成後FIT（元/度）,屋頂型_100瓩以上不及500瓩_加成後FIT（元/度）,屋頂型_10瓩以上不及20瓩_加成後FIT（元/度）,屋頂型_1瓩以上不及10瓩_加成後FIT（元/度）,屋頂型_20瓩以上不及50瓩_加成後FIT（元/度）,屋頂型_500瓩以上_加成後FIT（元/度）,屋頂型_50瓩以上不及100瓩_加成後FIT（元/度）,水面型_1瓩以上_加成後FIT（元/度）
0,9007010,連江縣,南竿鄉,4.397411,4.5981,6.497683,6.577428,5.050472,4.522789,4.975094,4.850328
1,9007020,連江縣,北竿鄉,4.397411,4.5981,6.497683,6.577428,5.050472,4.522789,4.975094,4.850328
2,9007030,連江縣,莒光鄉,4.397411,4.5981,6.497683,6.577428,5.050472,4.522789,4.975094,4.850328


## 6. Validation and summary

In [7]:
# %%
# 6. Validation — missing values

print('=== Output shape ===')
print(f'Rows: {len(policy_features)} (expected 368)')
print(f'Columns: {len(policy_features.columns)} = 3 ID + {len(fit_feature_cols)} FIT')

miss = policy_features[fit_feature_cols].isna().sum()
print('\n=== Missing values ===')
if miss.sum() == 0:
    print('No missing values in FIT columns.')
else:
    print(pd.DataFrame({'missing_count': miss, 'missing_ratio': miss / len(policy_features)}))

=== Output shape ===
Rows: 368 (expected 368)
Columns: 11 = 3 ID + 8 FIT

=== Missing values ===
No missing values in FIT columns.


In [8]:
# County coverage: each county should map to all its townships

n_counties = policy_features['COUNTYNAME'].nunique()
print(f'Counties represented: {n_counties} (expected 22)')
print(policy_features.groupby('COUNTYNAME').size().describe())

Counties represented: 22 (expected 22)
count    22.000000
mean     16.727273
std      10.959587
min       2.000000
25%       8.250000
50%      13.000000
75%      24.500000
max      38.000000
dtype: float64


In [9]:
# Annual trend check (text only; no plot — avoid font issues on headless/macOS)

ref_type, ref_cap = '屋頂型', '1瓩以上不及10瓩'
yearly_fit = (
    fit_clean
    .query('solar_type == @ref_type and capacity_category == @ref_cap')
    .groupby('year')['feedin_tariff']
    .mean()
    .sort_index()
)

print(f'=== Yearly mean FIT ({ref_type} / {ref_cap}, after phase avg within year) ===')
print(yearly_fit.round(4).to_string())
if len(yearly_fit) >= 2:
    declined = yearly_fit.iloc[-1] < yearly_fit.iloc[0]
    print(f"\nDeclined {yearly_fit.index[0]} -> {yearly_fit.index[-1]}: {declined}")

=== Yearly mean FIT (屋頂型 / 1瓩以上不及10瓩, after phase avg within year) ===
year
2018    6.2478
2019    5.9999
2020    6.2014
2021    6.1322
2022    6.3391
2023    6.3391
2024    6.2361
2025    6.1509
2026    6.1088

Declined 2018 -> 2026: True


In [10]:
# Top / bottom counties (representative column after full aggregation)

rep_col = make_fit_feature_col('屋頂型', '1瓩以上不及10瓩')
county_rep = policy_features.groupby('COUNTYNAME')[rep_col].first().sort_values(ascending=False)

print(f'=== {rep_col} — Top 5 counties ===')
print(county_rep.head(5).round(4).to_string())
print(f'\n=== {rep_col} — Bottom 5 counties ===')
print(county_rep.tail(5).round(4).to_string())

=== 屋頂型_1瓩以上不及10瓩_加成後FIT（元/度） — Top 5 counties ===
COUNTYNAME
澎湖縣    6.5774
基隆市    6.5774
宜蘭縣    6.5774
金門縣    6.5774
連江縣    6.5774

=== 屋頂型_1瓩以上不及10瓩_加成後FIT（元/度） — Bottom 5 counties ===
COUNTYNAME
嘉義市    5.7195
彰化縣    5.7195
屏東縣    5.7195
嘉義縣    5.7195
高雄市    5.7195


## 7. Export

In [11]:
# %%
# 7. Export

policy_features.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

print('Saved:', OUT_CSV)
print('Shape:', policy_features.shape)
print('\nSample rows:')
print(policy_features[ID_COLS + fit_feature_cols[:3]].head(5).to_string(index=False))
print('\nFIT feature engineering completed.')

Saved: /Users/tingting/SolarPotentialMapping/data/taiwan/policy/taiwan_policy_feature.csv
Shape: (368, 11)

Sample rows:
 TOWNCODE COUNTYNAME TOWNNAME  地面型_1瓩以上_加成後FIT（元/度）  屋頂型_100瓩以上不及500瓩_加成後FIT（元/度）  屋頂型_10瓩以上不及20瓩_加成後FIT（元/度）
  9007010        連江縣      南竿鄉              4.397411                        4.5981                    6.497683
  9007020        連江縣      北竿鄉              4.397411                        4.5981                    6.497683
  9007030        連江縣      莒光鄉              4.397411                        4.5981                    6.497683
  9007040        連江縣      東引鄉              4.397411                        4.5981                    6.497683
  9020010        金門縣      金城鎮              4.397411                        4.5981                    6.497683

FIT feature engineering completed.


## 總結

- **唯一輸出**：`data/taiwan/policy/taiwan_policy_feature.csv`
- **列**：368 鄉鎮（`TOWNCODE` 唯一鍵）
- **欄**：`TOWNCODE`, `COUNTYNAME`, `TOWNNAME` + 8 個 `{類型}_{級距}_加成後FIT（元/度）`（113 年制，107–112 舊制已展開對照）
- **與 DeepSolar**：美國 `feedin_tariff` 為政策年數；台灣為歷年平均躉購費率，語意相近但不可直接比較數值尺度
- **交接文件**：`data/taiwan/policy/04e_taiwan_policy_summary.md`